# 全部測站&平均  pffr ffpc

In [1]:
%load_ext rpy2.ipython

In [2]:
import os, math, random
from typing import Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# rpy2
import rpy2.robjects as ro
from rpy2.robjects import default_converter
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import numpy2ri

In [ ]:
%%R
TEST_K <- 365
TEST_START_DATE <- as.Date("2025-1-01")
NONNEGATIVE <- TRUE

# Total prediction horizon: 72h, rolling window: 24h each
HORIZON <- 72
WINDOW <- 24
N_ROLLS <- HORIZON / WINDOW

# Path to FPCA data (for model input) and original Y data (for evaluation)
csv_path <- "完整data/FPCA_PM25_format_all.csv"
orig_csv_path <- "完整data/merged_reshaped_PM2.5.csv"

# ffpc 參數
s.n.basis <- 30
t.n.basis <- 15
u.n.basis <- 15
upper.comp <- 12
time_grid <- 0:23

# -------- 套件 --------
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(lubridate)
  library(ggplot2)
  library(stringr)
  library(tibble)
  library(fda)
  library(refund)
})

# ---------------- 工具函數 ----------------
build_daily_matrix <- function(dates, values, min_obs_per_day = 5) {
  unique_days <- sort(unique(dates))
  pm25_matrix <- matrix(NA_real_, nrow = length(unique_days), ncol = 24)
  
  for (i in seq_along(unique_days)) {
    day_mask <- dates == unique_days[i]
    day_vals <- values[day_mask]
    day_hours <- as.integer(format(unique_days[i], "%H"))
    if (length(day_hours) < length(day_vals)) {
      day_hours <- seq_len(length(day_vals)) - 1
    }
    for (j in seq_along(day_vals)) {
      h <- day_hours[1] + j
      if (h <= 24) pm25_matrix[i, h] <- day_vals[j]
    }
  }
  
  valid_days <- which(rowSums(!is.na(pm25_matrix)) >= min_obs_per_day)
  list(matrix = pm25_matrix[valid_days, , drop = FALSE],
       dates = unique_days[valid_days])
}

evaluate_predictions <- function(Y_pred, Y_raw) {
  mae <- numeric(nrow(Y_pred))
  rmse <- numeric(nrow(Y_pred))
  
  for (k in seq_len(nrow(Y_pred))) {
    pred <- as.numeric(Y_pred[k, ])
    y_raw <- as.numeric(Y_raw[k, ])
    valid_mask <- !is.na(y_raw)
    n_valid <- sum(valid_mask)
    
    if (n_valid == 0) {
      mae[k] <- NA_real_
      rmse[k] <- NA_real_
    } else {
      pred_valid <- pred[valid_mask]
      y_valid <- y_raw[valid_mask]
      mae[k] <- mean(abs(pred_valid - y_valid))
      rmse[k] <- sqrt(mean((pred_valid - y_valid)^2))
    }
  }
  
  list(mean_mae = mean(mae, na.rm = TRUE),
       mean_rmse = mean(rmse, na.rm = TRUE))
}

build_orig_Y <- function(pair_dates, orig_matrix, orig_date_info, N_ROLLS) {
  orig_date_to_row <- setNames(seq_len(nrow(orig_matrix)), as.character(orig_date_info))
  n_pairs <- length(pair_dates)
  Y_orig <- vector("list", N_ROLLS)
  
  for (r in 1:N_ROLLS) {
    Y_orig[[r]] <- matrix(NA_real_, nrow = n_pairs, ncol = 24)
    for (d in seq_len(n_pairs)) {
      target_date <- pair_dates[d] + r
      target_key <- as.character(target_date)
      if (target_key %in% names(orig_date_to_row)) {
        Y_orig[[r]][d, ] <- orig_matrix[orig_date_to_row[[target_key]], ]
      }
    }
  }
  
  return(Y_orig)
}

process_single_station <- function(fpca_values, fpca_dates, orig_values, orig_dates, station_name) {
  tryCatch({
    # Build daily matrices from FPCA (for model input X)
    fpca_daily <- build_daily_matrix(fpca_dates, fpca_values)
    fpca_matrix <- fpca_daily$matrix
    fpca_date_info <- fpca_daily$dates
    n_fpca_days <- nrow(fpca_matrix)
    
    # Build daily matrices from original Y (for evaluation)
    orig_daily <- build_daily_matrix(orig_dates, orig_values)
    orig_matrix <- orig_daily$matrix
    orig_date_info <- orig_daily$dates
    
    if (n_fpca_days < 100 + N_ROLLS) {
      return(list(station = station_name, error = "Insufficient FPCA data"))
    }
    
    # Create X (input) and Y (target) pairs from FPCA data
    X_all <- fpca_matrix[1:(n_fpca_days - N_ROLLS), , drop = FALSE]
    pair_dates <- fpca_date_info[1:(n_fpca_days - N_ROLLS)]
    
    # FPCA Y for training: day t -> day t+1
    Y_train_fpca <- fpca_matrix[2:(n_fpca_days - N_ROLLS + 1), , drop = FALSE]
    
    # Original Y for evaluation (at all horizons)
    Y_orig <- build_orig_Y(pair_dates, orig_matrix, orig_date_info, N_ROLLS)
    
    # Train/test split
    d_to_idx <- setNames(1:length(pair_dates), as.character(pair_dates))
    test_idx <- c()
    curD <- TEST_START_DATE
    last_date <- max(pair_dates)
    
    while (length(test_idx) < TEST_K && curD <= last_date) {
      key <- as.character(curD)
      if (key %in% names(d_to_idx)) test_idx <- c(test_idx, d_to_idx[[key]])
      curD <- curD + 1
    }
    
    if (length(test_idx) == 0) {
      return(list(station = station_name, error = "No test samples"))
    }
    
    test_idx <- sort(unique(test_idx))
    train_idx <- setdiff(1:nrow(X_all), test_idx)
    train_idx <- train_idx[train_idx < min(test_idx)]
    
    if (length(train_idx) < 50) {
      return(list(station = station_name, error = "Insufficient training samples"))
    }
    
    X_train <- X_all[train_idx, , drop = FALSE]
    Y_train <- Y_train_fpca[train_idx, , drop = FALSE]
    X_test  <- X_all[test_idx, , drop = FALSE]
    
    Y_test_orig <- vector("list", N_ROLLS)
    for (r in 1:N_ROLLS) {
      Y_test_orig[[r]] <- Y_orig[[r]][test_idx, , drop = FALSE]
    }
    
    # Train pffr(ffpc) model
    npc_ffpc <- min(upper.comp, min(ncol(X_train), ncol(Y_train)))
    
    t0 <- proc.time()[3]
    fit.pffr <- pffr(
      Y_train ~ ffpc(X_train, xind = time_grid, yind = time_grid, npc = npc_ffpc),
      yind = time_grid,
      algorithm = "bam",
      method = "REML"
    )
    training_time <- proc.time()[3] - t0
    
    # Rolling 72h prediction (3 rounds x 24h)
    t0 <- proc.time()[3]
    n_test <- nrow(X_test)
    Y_pred <- vector("list", N_ROLLS)
    
    for (i in 1:n_test) {
      X_curr <- X_test[i, , drop = FALSE]
      for (r in 1:N_ROLLS) {
        pred <- predict(fit.pffr, newdata = list(X_train = X_curr), yind = time_grid)
        if (NONNEGATIVE) pred <- pmax(pred, 0)
        if (is.null(Y_pred[[r]])) {
          Y_pred[[r]] <- matrix(NA_real_, nrow = n_test, ncol = 24)
        }
        Y_pred[[r]][i, ] <- as.numeric(pred)
        X_curr <- matrix(as.numeric(pred), nrow = 1)
      }
    }
    prediction_time <- proc.time()[3] - t0
    
    # Evaluate predictions against ORIGINAL Y
    mae_h <- numeric(N_ROLLS)
    rmse_h <- numeric(N_ROLLS)
    
    for (r in 1:N_ROLLS) {
      valid_rows <- which(rowSums(!is.na(Y_test_orig[[r]])) > 0)
      if (length(valid_rows) > 0) {
        ev <- evaluate_predictions(
          Y_pred[[r]][valid_rows, , drop = FALSE],
          Y_test_orig[[r]][valid_rows, , drop = FALSE]
        )
        mae_h[r] <- ev$mean_mae
        rmse_h[r] <- ev$mean_rmse
      } else {
        mae_h[r] <- NA_real_
        rmse_h[r] <- NA_real_
      }
    }
    
    avg_mae_3d <- mean(mae_h, na.rm = TRUE)
    avg_rmse_3d <- mean(rmse_h, na.rm = TRUE)
    
    return(list(
      station = station_name,
      n_train = length(train_idx),
      n_test = length(test_idx),
      ffpc_npc = npc_ffpc,
      training_time = training_time,
      prediction_time = prediction_time,
      mae_24h = mae_h[1], mae_48h = mae_h[2], mae_72h = mae_h[3],
      rmse_24h = rmse_h[1], rmse_48h = rmse_h[2], rmse_72h = rmse_h[3],
      avg_mae_3d = avg_mae_3d,
      avg_rmse_3d = avg_rmse_3d,
      error = NA_character_
    ))
    
  }, error = function(e) {
    return(list(station = station_name, error = as.character(e$message),
                avg_mae_3d = NA_real_, avg_rmse_3d = NA_real_))
  })
}

# ================= 主流程 =================
cat("\n========================================\n")
cat("  PM2.5 預測 - pffr(ffpc) vs 原始 Y\n")
cat("  Rolling 72h 預測 (每24h滾動,共3次)\n")
cat("========================================\n\n")

# 載入 FPCA 資料
cat(sprintf("Loading FPCA data from: %s\n", csv_path))
df_fpca <- read.csv(csv_path, stringsAsFactors = FALSE, fileEncoding = "UTF-8")
fpca_time <- as.POSIXct(df_fpca[[1]], tz = "UTC")
fpca_time_dates <- as.Date(fpca_time)
fpca_station_cols <- setdiff(names(df_fpca), names(df_fpca)[1])

# 載入原始 Y 資料 (用於評估)
cat(sprintf("Loading original Y data from: %s\n", orig_csv_path))
df_orig <- read.csv(orig_csv_path, stringsAsFactors = FALSE, fileEncoding = "UTF-8")
orig_time <- as.POSIXct(df_orig[[1]], tz = "UTC")
orig_time_dates <- as.Date(orig_time)
orig_station_cols <- setdiff(names(df_orig), names(df_orig)[1])

# 找出共同測站
common_stations <- intersect(fpca_station_cols, orig_station_cols)
cat(sprintf("FPCA stations: %d, Original stations: %d, Common: %d\n\n",
            length(fpca_station_cols), length(orig_station_cols), length(common_stations)))

results_list <- list()

for (i in seq_along(common_stations)) {
  station_name <- common_stations[i]
  cat(sprintf("[%d/%d] Processing station: %s\n", i, length(common_stations), station_name))
  
  fpca_values <- suppressWarnings(as.numeric(df_fpca[[station_name]]))
  orig_values <- suppressWarnings(as.numeric(df_orig[[station_name]]))
  
  if (all(is.na(fpca_values)) || all(is.na(orig_values))) {
    cat(sprintf("  Skipped: No data\n\n"))
    results_list[[i]] <- list(station = station_name, error = "No data",
                              avg_mae_3d = NA_real_, avg_rmse_3d = NA_real_)
    next
  }
  
  result <- process_single_station(fpca_values, fpca_time_dates,
                                    orig_values, orig_time_dates, station_name)
  results_list[[i]] <- result
  
  if (!is.na(result$avg_mae_3d)) {
    cat(sprintf("  MAE(24h)=%.3f, MAE(48h)=%.3f, MAE(72h)=%.3f\n",
                result$mae_24h, result$mae_48h, result$mae_72h))
    cat(sprintf("  Avg3d MAE=%.3f, Avg3d RMSE=%.3f  (n_train=%d, n_test=%d, time=%.1fs)\n\n",
                result$avg_mae_3d, result$avg_rmse_3d,
                result$n_train, result$n_test,
                result$training_time + result$prediction_time))
  } else {
    cat(sprintf("  Error: %s\n\n", result$error))
  }
}

# ================= 彙整結果 =================
cat("\n========================================\n")
cat("  Results Summary (vs Original Y)\n")
cat("========================================\n\n")

results_df <- dplyr::bind_rows(results_list)

success_df <- results_df %>%
  dplyr::filter(!is.na(avg_mae_3d))

failed_df <- results_df %>%
  dplyr::filter(is.na(avg_mae_3d))

cat(sprintf("Total stations: %d\n", nrow(results_df)))
cat(sprintf("Successful: %d\n", nrow(success_df)))
cat(sprintf("Failed: %d\n\n", nrow(failed_df)))

if (nrow(success_df) > 0) {
  cat("========================================\n")
  cat("  Per Station 3-Day Average\n")
  cat("========================================\n")
  for (i in 1:nrow(success_df)) {
    cat(sprintf("%s: Avg3d MAE=%.3f, Avg3d RMSE=%.3f\n",
                success_df$station[i], success_df$avg_mae_3d[i], success_df$avg_rmse_3d[i]))
  }
  
  cat("\n========================================\n")
  cat(sprintf("  Overall Average (all %d stations)\n", nrow(success_df)))
  cat("========================================\n")
  cat(sprintf("Mean Avg3d MAE  = %.3f\n", mean(success_df$avg_mae_3d, na.rm = TRUE)))
  cat(sprintf("Mean Avg3d RMSE = %.3f\n", mean(success_df$avg_rmse_3d, na.rm = TRUE)))
  
  cat(sprintf("\nPer-horizon overall means:\n"))
  cat(sprintf("  MAE(24h)=%.3f, RMSE(24h)=%.3f\n",
              mean(success_df$mae_24h, na.rm = TRUE), mean(success_df$rmse_24h, na.rm = TRUE)))
  cat(sprintf("  MAE(48h)=%.3f, RMSE(48h)=%.3f\n",
              mean(success_df$mae_48h, na.rm = TRUE), mean(success_df$rmse_48h, na.rm = TRUE)))
  cat(sprintf("  MAE(72h)=%.3f, RMSE(72h)=%.3f\n",
              mean(success_df$mae_72h, na.rm = TRUE), mean(success_df$rmse_72h, na.rm = TRUE)))
}

if (nrow(failed_df) > 0) {
  cat("\n========================================\n")
  cat("Failed Stations:\n")
  cat("========================================\n")
  for (i in 1:nrow(failed_df)) {
    cat(sprintf("%s: %s\n", failed_df$station[i], failed_df$error[i]))
  }
}

# 儲存結果
cat("\n========================================\n")
cat("Saving results...\n")
write.csv(results_df, "all_stations_results_ffpc_rolling72h_vs_origY.csv", row.names = FALSE)
cat("Results saved to: all_stations_results_ffpc_rolling72h_vs_origY.csv\n")

cat("\nDone.\n")
cat("========================================\n")